# Classify all beach cells with model-and-class weighted probability voting

This notebook loads the four saved beach-surface backbones and the validation-derived weights in `Results/bs_ensemble/<setup>/ensemble_definition.json` under `weights_by_model_and_class`. Each class has a separate weight for every model, calculated from validation class-specific F1 scores. This method is fixed explicitly; test scores are not used to fit weights.

Each backbone retains its saved context radius, normalization statistics, and image size. NaN/-999 and unavailable same-survey context cells use that model's training-channel means. No training or copying of raw matrices is performed.

Dates are processed sequentially. The raw date directory is scanned once for candidate targets; target completeness is checked directly and neighbours are read on demand. Weighted probabilities are accumulated one model at a time. Progress messages use flush=True for nohup execution.

Results are saved separately from global weighting to `Results/bs_all_cells_classification/weighted_ensemble_per_class/<setup>/<date>/classification.csv`, with exactly three columns: `date`, `cell_id`, and numeric `predicted_class`. A summary and the exact weight definition used are also saved under the output root.

Timing is reported in seconds for every backbone and survey. Backbone prediction includes matrix/context reading, preprocessing, data loading, GPU transfers, forward passes, softmax, and collecting probabilities; checkpoint loading is excluded. CUDA is synchronized at timing boundaries. Ensemble calculation includes weighting and accumulation across models, probability normalization, and final class selection/table construction, but excludes backbone prediction and CSV writing. Survey total includes discovery, all predictions, ensemble calculation, and prediction CSV writing, but excludes timing-report writing. Per-survey `prediction_timing.csv` and the incrementally updated root `classification_summary.csv` preserve timings after each completed survey. `run_timing.json` records completed-run wall time, including model loading.

## Channel-ablation setup
Select `RGB_only`, `RGB_slope`, `RGB_elevation`, or `RGB_slope_elevation` with `BEACH_CHANNEL_SETUP` (default: `RGB_slope_elevation`). Training and ensemble evaluation are independent for each setup, retaining identical splits/CV plans, parameter grids, and seeds. Only selected channels are read and normalized; the model input-channel count and saved metadata match the selection. Results are stored in the corresponding setup subfolder. Existing outputs below are retained and may predate this change; rerun to obtain setup-specific results.


In [ ]:
from pathlib import Path
import json
from time import perf_counter
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.channel_setups import get_channel_setup, setup_result_dir, validate_setup_metadata
from Source.ensemble import (
    classify_date_directories_with_class_ensemble,
    load_class_weighted_ensemble,
    load_model_artifact,
    validate_artifact_compatibility,
)
from Source.inference import discover_date_directories
from Source.training import get_device

In [ ]:
# Select a setup here, or export BEACH_CHANNEL_SETUP before batch execution.
CHANNEL_SETUP, FEATURE_SUFFIXES = get_channel_setup()
print(f"Channel setup: {CHANNEL_SETUP}; features: {FEATURE_SUFFIXES}", flush=True)

run_started = perf_counter()
SOURCE_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
ENSEMBLE_RESULTS_DIR = setup_result_dir(PROJECT_ROOT / "Results" / "bs_ensemble", CHANNEL_SETUP)
ENSEMBLE_DEFINITION_PATH = ENSEMBLE_RESULTS_DIR / "ensemble_definition.json"
OUTPUT_ROOT = setup_result_dir(PROJECT_ROOT / "Results" / "bs_all_cells_classification" / "weighted_ensemble_per_class", CHANNEL_SETUP)

MODEL_RESULT_DIRS = {
    "convnext_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "bs_convnext_tiny", CHANNEL_SETUP),
    "vit_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "bs_vit_tiny", CHANNEL_SETUP),
    "swin_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "bs_swin_tiny", CHANNEL_SETUP),
    "efficientnetv2_s": setup_result_dir(PROJECT_ROOT / "Results" / "bs_efficientnetv2_s", CHANNEL_SETUP),
}

BATCH_SIZE = 64
NUM_WORKERS = 0
BASE_SUFFIX = "r"

print(f"Project root:       {PROJECT_ROOT}", flush=True)
print(f"Source root:        {SOURCE_ROOT}", flush=True)
print(f"Ensemble definition: {ENSEMBLE_DEFINITION_PATH}", flush=True)
print(f"Output root:        {OUTPUT_ROOT}", flush=True)
ENSEMBLE_METHOD = "weighted_probability_per_model_class"
print(f"Inference method: {ENSEMBLE_METHOD}", flush=True)

In [ ]:
print("Checking inputs and model artifacts...", flush=True)
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Cell data directory not found: {SOURCE_ROOT}")
if not ENSEMBLE_DEFINITION_PATH.exists():
    raise FileNotFoundError(
        f"Ensemble definition not found: {ENSEMBLE_DEFINITION_PATH}. "
        "Run bs_evaluate_ensemble.ipynb first."
    )

artifacts = [
    load_model_artifact(name, result_dir)
    for name, result_dir in MODEL_RESULT_DIRS.items()
]
validate_artifact_compatibility(artifacts)
for artifact in artifacts:
    validate_setup_metadata(artifact.metadata, CHANNEL_SETUP)
reference_metadata = artifacts[0].metadata
idx_to_class_code = {
    int(index): str(code)
    for index, code in reference_metadata["idx_to_class_code"].items()
}

artifact_table = pd.DataFrame([
    {
        "model": artifact.name,
        "context_radius": int(artifact.metadata["context_radius"]),
        "image_size": int(artifact.metadata["image_size"]),
        "checkpoint": str(artifact.checkpoint_path),
    }
    for artifact in artifacts
])
artifact_table

In [ ]:
print("Selecting device and loading the four best backbones with class-specific weights...", flush=True)
device = get_device()
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(device)}", flush=True)

models, model_class_weights, ensemble_definition = load_class_weighted_ensemble(
    artifacts,
    ENSEMBLE_DEFINITION_PATH,
    device,
)
weight_table = pd.DataFrame(
    {artifact.name: model_class_weights[artifact.name] for artifact in artifacts},
    index=pd.Index([idx_to_class_code[index] for index in range(len(idx_to_class_code))], name="class_code"),
)
print("Loaded weights (each row sums to one across models):", flush=True)
display(weight_table.style.format("{:.6f}"))
print(f"Inference explicitly uses: {ENSEMBLE_METHOD}", flush=True)
print(f"Validation-selected method recorded in definition: {ensemble_definition.get('validation_selected_method')}", flush=True)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
weight_table.to_csv(OUTPUT_ROOT / "model_class_weights.csv")
with (OUTPUT_ROOT / "inference_ensemble_definition.json").open("w", encoding="utf-8") as file:
    json.dump({
        "inference_method": ENSEMBLE_METHOD,
        "source_definition": str(ENSEMBLE_DEFINITION_PATH),
        "source_ensemble_definition": ensemble_definition,
        "normalized_weights_by_model_and_class": {
            name: {idx_to_class_code[index]: float(weight) for index, weight in enumerate(weights)}
            for name, weights in model_class_weights.items()
        },
        "model_metadata": {artifact.name: artifact.metadata for artifact in artifacts},
    }, file, indent=2)

In [ ]:
print("Discovering date folders...", flush=True)
date_dirs = discover_date_directories(SOURCE_ROOT)
print(f"Date folders found: {len(date_dirs)}", flush=True)
print(
    "Each date is scanned once for complete target cells. "
    "Neighbor files are then read on demand using each model's context radius.",
    flush=True,
)

In [ ]:
print("Starting model-and-class weighted ensemble classification...", flush=True)
classification_summary = classify_date_directories_with_class_ensemble(
    date_dirs=date_dirs,
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    artifacts=artifacts,
    models=models,
    weights=model_class_weights,
    idx_to_class_code=idx_to_class_code,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    base_suffix=BASE_SUFFIX,
    device=device,
)
classification_summary.to_csv(OUTPUT_ROOT / "classification_summary.csv", index=False)
run_wall_seconds = perf_counter() - run_started
with (OUTPUT_ROOT / "run_timing.json").open("w", encoding="utf-8") as file:
    json.dump({"run_wall_seconds": run_wall_seconds, "survey_total_seconds": float(classification_summary.get("survey_total_seconds", pd.Series(dtype=float)).sum()), "completed_surveys": len(classification_summary)}, file, indent=2)
print(f"Total run wall time (including model loading): {run_wall_seconds:.2f}s ({run_wall_seconds / 3600:.2f}h)", flush=True)
print("Model-and-class weighted classification finished.", flush=True)
print(f"Results: {OUTPUT_ROOT}", flush=True)
display(classification_summary)